In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score,balanced_accuracy_score, recall_score

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


### Load the embeddings for the C.S.sylv sulcal region

In [2]:
base_model_path = '/neurospin/dico/data/deep_folding/current/models/Champollion_V0'
list_model_path = [f'{base_model_path}_trained_on_UKB40/SC-sylv_right/11-36-10_85_0/ukb40_random_epoch80_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/11-43-38_3/ukb40_random_epoch100_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/13-19-08_28/ukb40_random_epoch80_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/13-35-41_0/ukb40_random_epoch100_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/13-35-41_1/ukb40_random_epoch100_embeddings/full_embeddings.csv',
                   f'{base_model_path}/SC-sylv_right/13-35-41_2/ukb40_random_epoch100_embeddings/full_embeddings.csv'

]
ukb_embeddings = pd.read_csv(list_model_path[5], index_col=0) 
print(ukb_embeddings.shape)
ukb_embeddings.head()

(42433, 256)


,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,dim9,dim10,...,dim247,dim248,dim249,dim250,dim251,dim252,dim253,dim254,dim255,dim256
ID,,,,,,,,,,,,,,,,,,,,,
sub-1000021,13.560396,-19.186274,9.990884,-18.254286,100.743576,21.419376,9.992589,38.310966,10.182008,8.429997,...,-8.648395,9.241949,15.904559,0.969972,-18.191433,48.243920,-9.597712,-3.207888,-1.590315,48.819576
sub-1000325,6.814707,5.649166,3.206304,-11.504702,-47.470943,-1.752250,14.649823,107.127680,-15.549886,-74.712204,...,9.333509,19.870533,43.917503,17.626700,16.755590,-21.635767,2.050762,-15.147051,-67.723960,50.983547
sub-1000458,-32.153885,-15.852353,-34.261200,-25.440128,155.755100,42.754630,28.028788,25.317122,-73.077060,22.907522,...,7.215434,-5.735838,39.347300,-74.891800,-28.349146,48.660385,16.774841,3.573832,20.597977,19.810497
sub-1000575,-5.093153,-2.633122,-6.500848,5.643069,113.905490,17.865131,13.420020,52.410187,-26.657318,34.222270,...,9.967592,-19.176190,12.513085,-55.806950,-4.351985,19.711640,-9.363625,7.304453,51.987846,61.707160
sub-1000606,18.587187,-0.891408,28.452131,5.302750,114.144330,20.014860,46.185270,15.269918,-41.947628,-13.753301,...,20.637644,-1.659880,79.245800,21.695093,-35.979572,45.812470,1.436629,-34.981160,-25.369762,13.995726


### Reduce dimension (hope to remove the noise) with a PCA

In [3]:
n_components=37

pca = PCA(n_components=n_components)
pca.fit(ukb_embeddings)
print(pca.explained_variance_ratio_)
(np.cumsum(pca.explained_variance_ratio_) < 0.999).sum()

[1.81594128e-01 1.32988776e-01 1.16325550e-01 1.07310987e-01
 1.00091535e-01 7.74769347e-02 7.00074412e-02 5.96548897e-02
 3.69019493e-02 2.93045558e-02 2.73610670e-02 1.94402046e-02
 9.63666413e-03 7.78177933e-03 6.40806415e-03 3.67553450e-03
 2.75939716e-03 1.69937093e-03 1.52249167e-03 1.32555847e-03
 1.14819960e-03 9.26906080e-04 6.83155540e-04 5.79767460e-04
 3.64297178e-04 3.16210214e-04 2.55775532e-04 2.23181039e-04
 2.07497238e-04 1.79594230e-04 1.54098031e-04 1.37644961e-04
 1.14074604e-04 1.01229248e-04 1.00507977e-04 8.78791671e-05
 8.05870771e-05]


37

In [4]:
ukb_pca_bdd = pca.transform(ukb_embeddings)

In [5]:
#scaler = StandardScaler()
#scaler.fit(ukb_embeddings)
#ukb_scl_bdd = scaler.transform(ukb_embeddings)
#ukb_scl_bdd

In [92]:
interrupted = [
'sub-1376904',
'sub-3694216',
'sub-1037052',
'sub-3250551',
'sub-5401486',
'sub-1499791', # good
'sub-2693192',
'sub-1633860',
'sub-3292254',
'sub-1613821',
'sub-2771619',
'sub-3159828',
'sub-4632483',
'sub-5936108',
'sub-1111996', # not sure
'sub-2846621', # good
'sub-2004479',
'sub-5236788',
'sub-3061407', # very good
'sub-5693167',
'sub-2155264', # very good
'sub-2444973', # very good
'sub-5245412', # good
'sub-5574911', # very good
'sub-2852894', # very good
'sub-1106033', # very good
'sub-5984646', # very good
'sub-5739487', # very good
'sub-3492298', # good
'sub-5712569', # not sure
'sub-2200121', # not sure
'sub-5638090', # good
'sub-4496792', # good
'sub-5129881', # good
'sub-1775041', # good
'sub-1094593', # good
'sub-1358401', # good
'sub-4354208', # very good
'sub-1428452', # good
'sub-5731125',
'sub-4995189', # very good
'sub-2762943', # very good
'sub-3386408', # not sure
'sub-5665554', # not sure
'sub-1130686', # good
'sub-2484762', # good
'sub-5186095', # good
'sub-5569356',
'sub-4762603', # not sure
'sub-2573795', # good
'sub-5315648',
'sub-2731992',
'sub-4949978',
'sub-2776534',
'sub-2298245',
'sub-2570335',
'sub-3258249', # not sure
'sub-1748817', # not sure
'sub-4203366', # very good
'sub-4184635', # very good
'sub-3947538', # not sure
'sub-2741631', # not sure
'sub-3492301',
'sub-4447456',
'sub-2373286', # not sure
'sub-4732282',
'sub-3293670',
'sub-2149638', # good
'sub-4625643', # not sure
'sub-4328267',
'sub-2589361',
'sub-4232003',
'sub-5456948',
'sub-4420000', # not sure
'sub-1553423', # not sure
'sub-1405899', # not sure
'sub-2550690', # not sure
'sub-2986522', # not sure
'sub-1698233', # not sure
'sub-4603077', # not sure
'sub-3428215',
'sub-1935008',
'sub-4589882',
'sub-2323818',
'sub-2230154',
'sub-1675253',
'sub-4059279',
'sub-4067363',
'sub-1322441',
'sub-1417407',
'sub-3733675',
'sub-5531350',
'sub-1369171',
'sub-1807186',
'sub-1267836',
'sub-3758439',
'sub-4652131',
'sub-1052521',
'sub-5949398',
'sub-3672666',
'sub-4754998',
'sub-3791185',
'sub-4587270',
'sub-4599903',
'sub-5617588',
'sub-1428212',
'sub-3911620',
'sub-4152006',
'sub-1864685',
'sub-5366951',
'sub-3679537',
'sub-5209589',
'sub-4211996',
'sub-3913796',
'sub-5777436',
'sub-4340260',
'sub-1132414',
'sub-5428293',
'sub-5406975',
'sub-4286421',
'sub-3253763',
'sub-1154509',
'sub-3500106',
'sub-4779638',
'sub-1107908',
'sub-4186067',
'sub-4916661',
'sub-3489143',
'sub-4212266',
'sub-2487105',
'sub-1569878',
'sub-5946014',
'sub-4561628',
'sub-5993265',
'sub-3494489',
'sub-1421577',
'sub-2916349',
'sub-4935525',
'sub-3694967',
'sub-3450882',
'sub-1233271',
'sub-2519386',
'sub-2512500',
'sub-1950476',
'sub-2662418',
'sub-3719519',
'sub-3450499',
'sub-2310242',
'sub-3842960',
'sub-5492484',
'sub-4862708',
'sub-4833497',
'sub-1500568', 
'sub-1939418', 
'sub-4906379', 
'sub-2115581', 
'sub-2614377',
'sub-1141211',
'sub-3642709',
'sub-3497149',
'sub-5592018',
'sub-2320673',
'sub-3868176',
'sub-1820397',
'sub-2239593',
'sub-2775840',
'sub-2347505',
'sub-4686469',
'sub-5493206',
'sub-3083655',
'sub-5062065',
'sub-2305535',
'sub-3671430',
'sub-1878869',
'sub-2508411',
'sub-5542529',
'sub-4904838',
'sub-5843983',
'sub-3560278',
'sub-4155325',
'sub-2908940',
'sub-2419686',
'sub-3067378',
'sub-3015676', 
'sub-3915494',
'sub-2892713', 
'sub-4829542', 
'sub-4536995',
'sub-2383298',
'sub-4153164',
'sub-4200848', 
'sub-4742060',
'sub-1461493', 
'sub-4097116', 
'sub-1171410', 
'sub-5840071',
'sub-5068144', 
'sub-3628020',
'sub-2495997',
'sub-4123265',
'sub-4177375',
'sub-2241143',
'sub-2573666',
'sub-4013152',
'sub-5607182',
'sub-4325584',
'sub-5436800',
'sub-4423951',
'sub-5497410',
'sub-4994326',
'sub-3002359',
'sub-4964520',
'sub-5673452',
'sub-5729360',
'sub-5541913',
'sub-5760693',
'sub-2992538',
'sub-1520771',
'sub-4835863',
'sub-5109324',
'sub-3540772',
'sub-4882028',
'sub-5941267',
'sub-4172149',
'sub-2255458',
'sub-1430256',
'sub-4108637',
'sub-2628215',
'sub-4575340',
'sub-4437069',
'sub-5694968',
'sub-5581941',
'sub-4192947',
'sub-1619277',
'sub-3821500',
'sub-3800570',
'sub-1015120',
'sub-1163316',
'sub-4682976', 
'sub-5249934',
'sub-1152270',
'sub-2521483',
'sub-5658334',
'sub-4633137',
'sub-1314348',
'sub-2479594',
'sub-1060820',
'sub-4748953',
'sub-5672776',
'sub-4710436',
'sub-1423045',
'sub-4875009',
'sub-4012523',
'sub-1542671',
'sub-5154603',
'sub-5963584',
'sub-1245156',
'sub-2230522',
'sub-5761769',
'sub-1675715',
'sub-3496974',
'sub-5823327',
'sub-3227475',
'sub-1516266',
'sub-2949059',
'sub-3369114',
'sub-1719698',
'sub-4549003',
'sub-1915832',
'sub-3031778',
'sub-4665000',
'sub-2924410',
]

ambiguous = [
'sub-1310920',
'sub-2863742',
'sub-1911266',
'sub-4217758',
'sub-5222070',
'sub-3794487',
'sub-1420697',
'sub-1425827',
'sub-3891499',
'sub-3572724',
'sub-1053493',
'sub-4875056',
'sub-1527779', 
'sub-4397096', 
'sub-3109923',
'sub-4860959',
'sub-2425148',
'sub-2077194',
'sub-1649070',
'sub-2005939',
'sub-2889389',
'sub-4520944',
'sub-4281714',
'sub-3692612', 
'sub-2379487', 
'sub-5137278',
'sub-4571621',
'sub-2461041',
'sub-1140601',
'sub-3439492',
'sub-3816101',
'sub-1452398',
'sub-5474299',
'sub-5080160',
'sub-3834564',
'sub-1372315',
'sub-5426258',
'sub-3313248',
'sub-3538950',
'sub-2642697',
'sub-5493039',
'sub-3272797',
'sub-3902778',
'sub-2929118', 
'sub-4257283',
'sub-5522199',
'sub-3604986',
'sub-5185480',
'sub-5623262',
'sub-5317805',
'sub-3297125',
'sub-2489075',
'sub-1647006',
]

not_interrupted = [
'sub-3943435',
'sub-4854284',
'sub-4558487',
'sub-1977658',
'sub-5161517',
'sub-2077690',
'sub-5937161',
'sub-4326429',
'sub-1377158',
'sub-3462570',
'sub-4342190',
'sub-4816666',
'sub-3884683',
'sub-1103646',
'sub-1167379',
'sub-1190643',
'sub-1273718',
'sub-1286007',
'sub-1298876',
'sub-1352284',
'sub-1398736',
'sub-1422413',
'sub-1465129',
'sub-1597706',
'sub-1701563',
'sub-1734788',
'sub-1979982',
'sub-1996092',
'sub-2036033',
'sub-2097565',
'sub-2118136',
'sub-2141551',
'sub-2193253',
'sub-2207793',
'sub-2228486',
'sub-2284024',
'sub-2337820',
'sub-2349203',
'sub-2389411',
'sub-2420937',
'sub-2427515',
'sub-2538754',
'sub-2583027',
'sub-2592717',
'sub-2733674',
'sub-2741815',
'sub-2792782',
'sub-2802489',
'sub-2816262',
'sub-2833426',
'sub-2834970',
'sub-2837393',
'sub-2946274',
'sub-2957401',
'sub-2968297',
'sub-2970418',
'sub-3008660',
'sub-3009279',
'sub-3013938',
'sub-3227039',
'sub-3234836',
'sub-3264612',
'sub-3333294',
'sub-3334219',
'sub-3379262',
'sub-3388080',
'sub-3388306',
'sub-3401499',
'sub-3453064',
'sub-3525594',
'sub-3529189',
'sub-3541105',
'sub-3603191',
'sub-3627711',
'sub-3670173',
'sub-3693543',
'sub-3721299',
'sub-3722413',
'sub-3765466',
'sub-3936967',
'sub-3992259',
'sub-3994474',
'sub-4016129',
'sub-4027732',
'sub-4116944',
'sub-4411765',
'sub-4420611',
'sub-4428393',
'sub-4491384',
'sub-4519441',
'sub-4536778',
'sub-4727825',
'sub-4741296',
'sub-4755899',
'sub-4787289',
'sub-4791977',
'sub-4805119',
'sub-4805237',
'sub-4834994',
'sub-4868991',
'sub-5027399',
'sub-5054716',
'sub-5082433',
'sub-5117110',
'sub-5123219',
'sub-5147403',
'sub-5217534',
'sub-5237880',
'sub-5292898',
'sub-5293703',
'sub-5319071',
'sub-5430535',
'sub-5437419',
'sub-5486726',
'sub-5561142',
'sub-5578922',
'sub-5581707',
'sub-5605784',
'sub-5643778',
'sub-5649675',
'sub-5686761',
'sub-5723111',
'sub-5729132',
'sub-5749108',
'sub-5754849',
'sub-5836983',
'sub-5864979',
'sub-5910947',
'sub-5966409',
'sub-5998652',
'sub-5357627',
'sub-2204575',
'sub-2839753',
'sub-5335727',
'sub-5782466',
'sub-4520082',
'sub-1004170',
'sub-4158073', 
'sub-5684893',
'sub-4359496',
'sub-2040983',
'sub-5575777',
'sub-1116938',
'sub-4189639',
'sub-4507392',
'sub-5085553',
'sub-5457081',
'sub-4831688',
'sub-3976041',
'sub-4057189',
'sub-4202490',
'sub-4844615',
'sub-4747425',
'sub-1008582',
'sub-4039492',
'sub-2969851', 
'sub-2830945', 
'sub-1711798', 
'sub-5120758',
'sub-4949491', 
'sub-4772821', 
'sub-4450785', 
'sub-1110891',
'sub-2920350',
'sub-5217882',
'sub-5747063',
'sub-5040811',
'sub-3143577',
'sub-1779953',
'sub-2112770',
'sub-2327257',
'sub-2406636',
'sub-3745349',
'sub-2968941', 
'sub-2611694',
'sub-2676227',
'sub-1164936',
'sub-4794448',
'sub-5711287',
'sub-1502163',
'sub-1546206',
'sub-4710850',
'sub-3735109',
'sub-2800235',
'sub-5115775',
'sub-3746225',
'sub-1416140',
'sub-5907102',
'sub-5989851',
'sub-3974444',
'sub-2212279',
'sub-4899126',
'sub-4518664', 
'sub-5828387',
'sub-1385217',
'sub-2442019',
'sub-1273391',
'sub-2302082', 
'sub-2814161',
'sub-3785229',
'sub-4945539',
'sub-4479433',
'sub-1930115',
'sub-1218580',
'sub-4880356',
'sub-5159616',
'sub-1809920',
'sub-4089007',
'sub-2127220',
'sub-4479596',
'sub-3276505',
'sub-1367494',
'sub-2480789',
'sub-3889542', 
'sub-5912354',
'sub-4593980',
'sub-1219530',
'sub-5751516',
'sub-4804513',
'sub-4006078',
'sub-1828965',
'sub-4025167',
'sub-2598789',
'sub-3344616',
'sub-4791222',
'sub-5224500',
'sub-1100724',
'sub-4690601',
'sub-3791953',
'sub-4661668',
'sub-4164110',
'sub-4431586',
'sub-1109550',
'sub-4606013',
'sub-2899381', 
'sub-2234711', 
'sub-3624620', 
'sub-2980957', 
'sub-4932490',
'sub-2258124', 
'sub-4119595', 
'sub-4404054', 
'sub-2368134', 
'sub-2496737',
'sub-3519440', 
'sub-4698380', 
'sub-1874775', 
'sub-3797977',
'sub-1340306',
'sub-1691707',
'sub-3314602',
'sub-5412919',
'sub-1529050'
] 

In [93]:
"""
Problem with:
[
'sub-3716267',
'sub-5417598',
'sub-5472164',
'sub-5894417',
'sub-1703355',
'sub-4271898',
]
"""

"\nProblem with:\n[\n'sub-3716267',\n'sub-5417598',\n'sub-5472164',\n'sub-5894417',\n'sub-1703355',\n'sub-4271898',\n]\n"

In [94]:
X = ukb_embeddings.loc[interrupted + not_interrupted]
y = [1 for i in range(len(interrupted))] + [0 for i in range(len(not_interrupted))]
X_pca = pca.transform(X)
len(interrupted), len(not_interrupted)

(272, 253)

In [95]:
print(len(set(interrupted)), len(interrupted), '\n')
print(len(set(not_interrupted)), len(not_interrupted), '\n')
print(set([x for x in interrupted if interrupted.count(x) > 1]))
print(set([x for x in not_interrupted if not_interrupted.count(x) > 1]))
print((set.intersection(set(interrupted), set(not_interrupted))))
print((set.intersection(set(interrupted), set(ambiguous))))
print((set.intersection(set(ambiguous), set(not_interrupted))))

my_labelled_df = pd.DataFrame({"ID":interrupted + not_interrupted, "Interruption":y})
my_labelled_df.to_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/interrupted_SC_labelled.csv', index=False)

272 272 

253 253 

set()
set()
set()
set()
set()


In [96]:
df_interrupted_julien = pd.read_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/interrupted_CS_julien.csv')
df_ambiguous_julien = pd.read_csv('/neurospin/dico/data/deep_folding/current/datasets/UkBioBank40/ambiguous_CS_julien.csv')


print('No interruption (me) VS interruption (Julien):', df_interrupted_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==0]).ID).sum(), '\n')
print('Interruption for both of us:',df_interrupted_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==1]).ID).sum(), '\n')
print("Interruption (Julien) I don't have as interruption:",len((df_interrupted_julien[~df_interrupted_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==1]).ID)])),'\n')
print("Length of interruption Julien's list:", len(df_interrupted_julien), '\n')

print('No interruption (me) VS ambiguous (Julien):',df_ambiguous_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==0]).ID).sum(), '\n')
print('Interruption (me) VS ambiguous (Julien):',df_ambiguous_julien.ID.isin((my_labelled_df[my_labelled_df.Interruption==1]).ID).sum(), '\n')

print('Ambiguous (me) VS interruption (Julien):',df_interrupted_julien.ID.isin(ambiguous).sum(), '\n')
print('Ambiguous (me) VS ambiguous (Julien):',df_ambiguous_julien.ID.isin(ambiguous).sum(), '\n')
print("Length of ambiguous Julien's list:", len(df_ambiguous_julien), '\n')

No interruption (me) VS interruption (Julien): 0 

Interruption for both of us: 40 

Interruption (Julien) I don't have as interruption: 12 

Length of interruption Julien's list: 52 

No interruption (me) VS ambiguous (Julien): 0 

Interruption (me) VS ambiguous (Julien): 2 

Ambiguous (me) VS interruption (Julien): 8 

Ambiguous (me) VS ambiguous (Julien): 3 

Length of ambiguous Julien's list: 35 



In [97]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.33, random_state=42)

#### linear SVC model

In [98]:
model = SVC(kernel='linear', probability=True,
            random_state=42,
            C=0.001, class_weight='balanced')

For model comparison, we calculate the ROC AUC (wihout PCA) with the previous 

In [99]:
for path_i in list_model_path:
    ukb_embeddings = pd.read_csv(path_i, index_col=0) 
    X = ukb_embeddings.loc[interrupted + not_interrupted]
    outputs = {}
    val_pred = cross_val_predict(model, X, y, cv=5)
    auc = roc_auc_score(y, val_pred)
    outputs['labels_pred'] = val_pred
    outputs['auc'] = auc
    outputs['balanced_accuracy_score'] = balanced_accuracy_score(y, val_pred)

    print(path_i)
    print('ROC AUC (cv=5):', "{:.3f}".format(outputs['auc']), '\n')

/neurospin/dico/data/deep_folding/current/models/Champollion_V0_trained_on_UKB40/SC-sylv_right/11-36-10_85_0/ukb40_random_epoch80_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.721 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/11-43-38_3/ukb40_random_epoch100_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.809 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/13-19-08_28/ukb40_random_epoch80_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.793 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/13-35-41_0/ukb40_random_epoch100_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.732 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/13-35-41_1/ukb40_random_epoch100_embeddings/full_embeddings.csv
ROC AUC (cv=5): 0.803 

/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/13-35-41_2/ukb40_random_epoch100_embeddings/full_embeddings.csv
ROC AUC (

In [43]:
outputs = {}
val_pred = cross_val_predict(model, X_pca, y, cv=5)
auc = roc_auc_score(y, val_pred)
outputs['labels_pred'] = val_pred
outputs['auc'] = auc
outputs['balanced_accuracy_score'] = balanced_accuracy_score(y, val_pred)

print('ROC AUC (cv=5):', "{:.3f}".format(outputs['auc']), '\n')
print('Balanced accuracys score (cv=5):', "{:.3f}".format(outputs['balanced_accuracy_score']))

ROC AUC (cv=5): 0.804 

Balanced accuracys score (cv=5): 0.804


In [44]:
model.fit(X_train_pca, y_train)

print('Recall (test):', "{:.3f}".format(recall_score(y_test, model.predict(X_test_pca))), '\n')
print('ROC AUC (test):', "{:.3f}".format(roc_auc_score(y_test ,model.predict_proba(X_test_pca)[:,1])), '\n')
print('Balanced accuracy score (test):', "{:.3f}".format(balanced_accuracy_score(y_test, model.predict(X_test_pca))), '\n')
model.fit(X_pca, y)

Recall (test): 0.859 

ROC AUC (test): 0.893 

Balanced accuracy score (test): 0.787 



SVC(C=0.001, class_weight='balanced', kernel='linear', probability=True,
    random_state=42)

In [45]:
prediction = pd.DataFrame({"ID" : list(ukb_embeddings.index),
              "Pred" : model.predict_proba(ukb_pca_bdd)[:,1]})
#prediction.to_csv('/volatile/ad279118/UKB/CentralSulcus/interruption_pred.csv', index=False)
prediction

,ID,Pred
0,sub-1000021,0.009658
1,sub-1000325,0.004038
2,sub-1000458,0.012727
3,sub-1000575,0.007091
4,sub-1000606,0.001421
...,...,...
42428,sub-6023847,0.007813
42429,sub-6024038,0.011302
42430,sub-6024150,0.067223
42431,sub-6024379,0.056318


In [46]:
print('Maximum probability of prediction among the interrupted C.S. :',"{:.3f}".format(prediction[prediction["ID"].isin(interrupted)].Pred.max()), '\n')
print('Mean probability of prediction among the interrupted C.S. :', "{:.3f}".format(prediction[prediction["ID"].isin(interrupted)].Pred.mean()), '\n')
prediction[prediction['ID']=='sub-2036033']

Maximum probability of prediction among the interrupted C.S. : 0.992 

Mean probability of prediction among the interrupted C.S. : 0.788 



,ID,Pred
8695,sub-2036033,0.025265


In [17]:
((prediction[~(prediction["ID"].isin(interrupted))]).sort_values(by="Pred")[-5:].ID).to_list()

['sub-3807447', 'sub-5080160', 'sub-2040983', 'sub-4146798', 'sub-5474299']

To compare to: 

category.tsv

| category_id | title                                         | availability | group_type | descript                                                                 | notes                                                                   |
|-------------|-----------------------------------------------|--------------|------------|--------------------------------------------------------------------------|-------------------------------------------------------------------------|
| 136         | Mental health                                 | 0            | 1          | Results of the on-line mental health self-assessment questionnaire issued in 2016. |                                                                         |
| 137         | Mental distress                               | 0            | 1          | Mental distress reported within the on-line mental health questionnaire. |                                                                         |
| 138         | Depression                                    | 0            | 1          | Depression reported within the on-line mental health questionnaire.      |                                                                         |
| 139         | Mania                                         | 0            | 1          | Mania reported within the on-line mental health questionnaire.           |                                                                         |
| 140         | Anxiety                                       | 0            | 1          | Anxiety reported within the on-line mental health questionnaire.         |                                                                         |
| 141         | Addictions                                    | 0            | 1          | Addictions reported within the on-line mental health questionnaire.      |                                                                         |
| 142         | Alcohol use                                   | 0            | 1          | Alcohol use reported within the on-line mental health questionnaire.     |                                                                         |
| 143         | Cannabis use                                  | 0            | 1          | Cannabis use reported within the on-line mental health questionnaire.    |                                                                         |
| 144         | Unusual and psychotic experiences             | 0            | 1          | Unusual and psychotic experiences reported within the on-line mental health questionnaire. |                                                                         |
| 145         | Traumatic events                              | 0            | 1          | Traumatic events reported within the on-line mental health questionnaire. |                                                                         |
| 146         | Self-harm behaviours                          | 0            | 1          | Self-harm behaviours reported within the on-line mental health questionnaire. |                                                                         |
| 147         | Happiness and subjective well-being           | 0            | 1          | Happiness and subjective well-being reported within the on-line mental health questionnaire. |                                                                         |
| 2415        | Pregnancy, childbirth and the puerperium      | 0            | 1          | First reported occurrences of conditions falling within the ICD10 classification Chapter XV Pregnancy, childbirth and the puerperium. |                                                                         |


encoding.tsv

| encoding_id | title                    | availability | coded_as | structure | num_members | descript                                                                   |
|-------------|--------------------------|--------------|----------|-----------|-------------|----------------------------------------------------------------------------|
| 100694      | Bipolar type             | 0            | 11       | 1         | 2           | Type of bipolar episode                                                    |
| 1405        | Depression substances     | 0            | 11       | 1         | 4           | Substances taken to potentially alleviate depressive symptoms.              |
| 1406        | Depression therapies      | 0            | 11       | 1         | 3           | Non-drug therapies aimed at alleviating depressive symptoms                  |
| 1908        | Antidepressant medications| 0            | 11       | 1         | 9           | Antidepressant medications                                                  |
| 3006        | Depression frequency      | 0            | 11       | 1         | 5           | Frequency of current depression symptoms with option of prefer not to answer |
| 100695      | Depressive episode        | 0            | 11       | 1         | 6           | Type of depressive episode(s)                                               |

field.tsv

| field_id | title                                         | availability | stability | private | value_type | base_type | item_type | strata | instanced | arrayed | sexed | units | main_category | encoding_id | instance_id | instance_min | instance_max | array_min | array_max | num_participants | item_count | showcase_order | cost_do | cost_on | cost_sc |
|----------|-----------------------------------------------|--------------|-----------|--------|------------|-----------|-----------|--------|-----------|---------|-------|-------|---------------|-------------|-------------|--------------|--------------|-----------|-----------|-----------------|------------|----------------|---------|---------|---------|
| 46       | Hand grip strength (left)      | 0            | 2         | 0      | 11         | 0         | 0         | 0      | 1         | 0       | 0     | Kg    | 100019        | 0           | 2           | 0            | 3            | 0         | 0         | Left grip strength. An issue has been identified with a small amount of grip strength data (~F46~ and ~F47~) collected in Cheadle during the first repeat visit in 2013 (Instance 1). | 2012-01-05T00:00:00 | 2023-10-01T00:00:00 | 499213           | 599524     | 1              | 1       | 1       | 1       |
| 47       | Hand grip strength (right)     | 0            | 2         | 0      | 11         | 0         | 0         | 0      | 1         | 0       | 0     | Kg    | 100019        | 0           | 2           | 0            | 3            | 0         | 0         | Right grip strength. An issue has been identified with a small amount of grip strength data (~F46~ and ~F47~) collected in Cheadle during the first repeat visit in 2013 (Instance 1). | 2012-01-05T00:00:00 | 2023-10-01T00:00:00 | 499291           | 599607     | 1.5            | 1       | 1       | 1       |
| 1707     | Handedness (chirality/laterality)             | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100033     | 100430      | 2            | 0            | 2         | 0         | 501463          | 533456     | 5              | 1       | 1       | 1       |
| 1920     | Mood swings                                   | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501480          | 604063     | 1              | 1       | 1       | 1       |
| 1930     | Miserableness                                 | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501480          | 604063     | 2              | 1       | 1       | 1       |
| 1940     | Irritability                                  | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501480          | 604063     | 3              | 1       | 1       | 1       |
| 1950     | Sensitivity / hurt feelings                   | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501479          | 604062     | 4              | 1       | 1       | 1       |
| 1960     | Fed-up feelings                               | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501479          | 604062     | 5              | 1       | 1       | 1       |
| 1970     | Nervous feelings                              | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501479          | 604062     | 6              | 1       | 1       | 1       |
| 1980     | Worrier / anxious feelings                    | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501478          | 604061     | 7              | 1       | 1       | 1       |
| 1990     | Tense / 'highly strung'                       | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501478          | 604061     | 8              | 1       | 1       | 1       |
| 2000     | Worry too long after embarrassment            | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501478          | 604061     | 9              | 1       | 1       | 1       |
| 2010     | Suffer from 'nerves'                          | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501478          | 604061     | 10             | 1       | 1       | 1       |
| 2020     | Loneliness, isolation                         | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501477          | 604060     | 11             | 1       | 1       | 1       |
| 2030     | Guilty feelings                               | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501477          | 604060     | 12             | 1       | 1       | 1       |
| 2040     | Risk taking                                   | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100349      | 2            | 0            | 3         | 0         | 501477          | 604060     | 13             | 1       | 1       | 1       |
| 2050     | Frequency of depressed mood in last 2 weeks   | 0            | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100484      | 2            | 0            | 3         | 0         | 501477          | 604060     | 20             | 1       | 1       | 1       |
| 2060     | Frequency of unenthusiasm / disinterest in last 2 weeks | 0 | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100484      | 2            | 0            | 3         | 0         | 501476          | 604059     | 21             | 1       | 1       | 1       |
| 2070     | Frequency of tenseness / restlessness in last 2 weeks | 0 | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100484      | 2            | 0            | 3         | 0         | 501475          | 604058     | 22             | 1       | 1       | 1       |
| 2080     | Frequency of tiredness / lethargy in last 2 weeks | 0 | 0         | 0      | 21         | 11        | 0         | 0      | 1         | 0       | 0     |       |               | 100060     | 100484      | 2            | 0            | 3         | 0         | 501474          | 604057     | 23             | 1       | 1       | 1       |
| 2090     | Seen doctor (GP) for nerves, anxiety, tension or depression | 0 | 0 | 0 | 21 | 11 | 0 | 0 | 1 | 0 | 0 | | 100060 | 100349 | 2 | 0 | 3 | 0 | 0 | 501473 | 604056 | 24 | 1 | 1 | 1 |
| 2100     | Seen a psychiatrist for nerves, anxiety, tension or depression | 0 | 0 | 0 | 21 | 11 | 0 | 0 | 1 | 0 | 0 | | 100060 | 100349 | 2 | 0 | 3 | 0 | 0 | 501473 | 604056 | 25 | 1 | 1 | 1 |



#### Second approach: Euclidian distance in the reduced latent space

In [18]:
from scipy.spatial import distance

In [521]:
list_dist = [distance.euclidean(pca.transform(ukb_embeddings.loc['sub-3791185'].to_numpy().reshape(1,-1)), ukb_pca_bdd[i]) for i in range(len(ukb_pca_bdd))]
df_dist = pd.DataFrame({"ID":list(ukb_embeddings.index), "Dist":list_dist})

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr

In [522]:
sample_dist = ((df_dist[~(df_dist["ID"].isin(interrupted))]).sort_values(by='Dist').iloc[22000:22025].ID).to_list()

### Visualization with Anatomist

In [19]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims

existing QApplication: 0
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-ad279118'


create qapp
global modules: /casa/host/build/share/anatomist-5.2/python_plugins
home   modules: /casa/home/.anatomist/python_plugins
done
Starting Anatomist.....
config file : /casa/home/.anatomist/config/settings.cfg
PyAnatomist Module present
PythonLauncher::runModules()
loading module simple_controls
loading module save_resampled
loading module selection
loading module bsa_proba
loading module modelGraphs
loading module profilewindow
loading module ana_image_math
loading module paletteViewer
loading module foldsplit
loading module anacontrolmenu
loading module gradientpalette
loading module palettecontrols
loading module meshsplit
loading module volumepalettes
loading module gltf_io
loading module infowindow
loading module histogram
loading module statsplotwindow
loading module valuesplotwindow
all python modules loaded
Anatomist started.


In [47]:
dataset = 'UkBioBank40'
region = "S.C.-sylv."
side = "R"

mm_skeleton_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops'

In [102]:
sample = ((prediction[~(prediction["ID"].isin(interrupted+ambiguous+not_interrupted))]).sort_values(by="Pred", ascending=False)[-2025:-2000].ID).to_list()

In [103]:
volume=True
volume_files = []

for subject_id in sample:
    volume_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"
    
    if volume:
        if os.path.isfile(volume_path):
            vol = aims.read(volume_path)
            volume_files.append(vol)
        else:
            print(f"{volume_path} is not a correct path, or the .nii.gz doesn't exist")

block = a.createWindowsBlock(5) # 10 columns
dic_windows = {}

if volume:
    for i, vol in enumerate(volume_files):
        dic_windows[f'a_vol{i}'] = a.toAObject(vol)
        #dic_windows[f'a_vol{i}'].setPalette(absoluteMode=True)
        dic_windows[f'rvol{i}'] = a.fusionObjects(objects=[dic_windows[f'a_vol{i}']], method='VolumeRenderingFusionMethod')
        dic_windows[f'rvol{i}'].releaseAppRef()
        dic_windows[f'wvr{i}'] = a.createWindow('3D', block=block) #geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
        dic_windows[f'wvr{i}'].addObjects(dic_windows[f'rvol{i}'])

no position could be read at 277, 153
no position could be read at 307, 89
no position could be read at 262, 70
no position could be read at 212, 106


In [91]:
sample[13]

'sub-2924410'

In [1077]:
sample_dist[1]

['sub-5293703', 'sub-5319071', 'sub-3525594', 'sub-5561142']

# Analysis

In [26]:
Birth_Weight = pd.read_csv('/volatile/ad279118/UKB/CentralSulcus/BirthWeight.csv')
print(Birth_Weight.shape, '\n')
print(Birth_Weight.columns,'\n')
print('Nb of Nan for "Birth weight | Instance 0":', Birth_Weight["participant.p20022_i0"].isna().sum())
print('Nb of Nan for "Birth weight | Instance 1":',Birth_Weight["participant.p20022_i1"].isna().sum())
print('Nb of Nan for "Birth weight | Instance 2":',Birth_Weight["participant.p20022_i2"].isna().sum())

Birth_Weight["Birth_Weight"] = Birth_Weight["participant.p20022_i0"]
Birth_Weight = Birth_Weight.drop(["participant.p20022_i0", "participant.p20022_i1", "participant.p20022_i2"], axis=1)
Birth_Weight["ID"] = Birth_Weight["0"].apply(lambda x : 'sub-'+str(x))
Birth_Weight = Birth_Weight.drop("0", axis=1)
Birth_Weight.head()

(42402, 4) 

Index(['0', 'participant.p20022_i0', 'participant.p20022_i1',
       'participant.p20022_i2'],
      dtype='object') 

Nb of Nan for "Birth weight | Instance 0": 16990
Nb of Nan for "Birth weight | Instance 1": 37970
Nb of Nan for "Birth weight | Instance 2": 38633


,Birth_Weight,ID
0,NaN,sub-1000021
1,0.94,sub-1000325
2,NaN,sub-1000458
3,2.38,sub-1000575
4,3.40,sub-1000606


In [48]:
merged = pd.merge(left=prediction, right=Birth_Weight, left_on='ID', right_on='ID', how='inner')
#merged = merged.drop('Participant ID', axis=1)
interrupted_with_BW = merged[merged.ID.isin(interrupted)].dropna()
not_interrupted_with_BW = merged[merged.ID.isin((prediction.sort_values(by='Pred')).iloc[1000:30000,:].ID)].dropna()
print(interrupted_with_BW[["Pred", "Birth_Weight"]].mean(axis=0),'\n')
print(not_interrupted_with_BW[["Pred", "Birth_Weight"]].mean(axis=0),'\n')

Pred            0.785347
Birth_Weight    3.394342
dtype: float64 

Pred            0.015546
Birth_Weight    3.353757
dtype: float64 

